In [1]:
import transformers
from transformers import AutoTokenizer , AutoModelForTokenClassification
from datasets import load_dataset
import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import get_scheduler
from tqdm.auto import tqdm
from seqeval.metrics import f1_score as ner_f1
from seqeval.metrics import classification_report


In [2]:
!pip install seqeval

In [3]:
dataset = load_dataset("wikiann", "en")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})


In [4]:
print(len(dataset['train']))
print(len(dataset['test']))

20000
10000


In [5]:
samp = dataset['train'][0]
print(samp['tokens'])
print(samp['ner_tags'])
print(samp['spans'])

['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')']
[3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0]
['ORG: R.H. Saunders', 'ORG: St. Lawrence River']


In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


In [7]:
def tokenizer_dataset(dataset):
    tokenized_inputs = tokenizer(dataset['tokens'], truncation=True, is_split_into_words=True , padding='max_length' , max_length=128)

    labels= []
    for i, label in enumerate(dataset['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs['labels'] = labels
    return tokenized_inputs



In [8]:
train_tokenized = dataset['train'].map(
    tokenizer_dataset,
    batched=True,
    remove_columns=dataset['train'].column_names
)
val_tokenized = dataset['test'].map(
    tokenizer_dataset,
    batched=True,
    remove_columns=dataset['test'].column_names
)

In [9]:
train_tokenized.set_format("torch")
val_tokenized.set_format("torch")

print(f"Train: {len(train_tokenized)}")
print(f"Val:   {len(val_tokenized)}")
print(f"Columns: {train_tokenized.column_names}")

Train: 20000
Val:   10000
Columns: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [10]:
train_loader = DataLoader(
    train_tokenized,
    batch_size=64,
    shuffle=True
)
val_loader = DataLoader(
    val_tokenized,
    batch_size=64,
    shuffle=False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

Train batches: 313
Val batches:   157


In [11]:
label_names = dataset['train'].features['ner_tags'].feature.names
num_labels  = len(label_names)

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

In [12]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [13]:
print(f"Num labels: {num_labels}")
print(f"Device:     {device}")
print(f"Labels:     {label_names}")

Num labels: 7
Device:     cuda
Labels:     ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5 , weight_decay=0.01)
num_epochs        = 3
num_training_steps = num_epochs * len(train_loader)
num_warmup_steps  = 100
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)
# do not use loss because the loss is measured automatically


In [15]:
train_tokenized.column_names

['input_ids', 'token_type_ids', 'attention_mask', 'labels']

In [16]:
# training
def training(model , loder , optimizer , scheduler , device):
  model.train()
  total_loss = 0
  num_batches = 0
  for batch_idx , batch in enumerate(loder):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)
    outputs = model(input_ids , attention_mask=attention_mask , labels=labels)
    loss = outputs.loss
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    total_loss += loss.item()
    num_batches += 1
    if (batch_idx + 1) % 10 == 0:
            avg_loss = total_loss / num_batches
            print(f"  Batch {batch_idx+1}/{len(loder)} | Loss: {avg_loss:.4f}")
  return total_loss / num_batches


In [17]:
def evaluate(model, loader, device, label_names):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss   = outputs.loss
            logits = outputs.logits
            total_loss += loss.item()

            predictions = torch.argmax(logits, dim=-1)
            preds_np    = predictions.cpu().numpy()
            labels_np   = labels.cpu().numpy()

            for pred_seq, label_seq in zip(preds_np, labels_np):
                pred_tags  = []
                label_tags = []

                for pred, label in zip(pred_seq, label_seq):
                    if label != -100:
                        pred = min(pred, len(label_names) - 1)
                        pred_tags.append(label_names[pred])
                        label_tags.append(label_names[label])

                if len(pred_tags) > 0:
                    all_preds.append(pred_tags)
                    all_labels.append(label_tags)

    f1       = ner_f1(all_labels, all_preds)
    avg_loss = total_loss / len(loader)

    return avg_loss, f1, all_preds, all_labels

In [18]:
# complete loop
print("-"*50)
print("  NER FINE-TUNING WITH  MANUAL TRAINING LOOP")   # wothout trainer
print("="*55)
print(f"Epochs:     {num_epochs}")
print(f"Train size: {len(train_tokenized)}")
print(f"Val size:   {len(val_tokenized)}")
print(f"Device:     {device}")
print("="*55)

--------------------------------------------------
  NER FINE-TUNING WITH  MANUAL TRAINING LOOP
Epochs:     3
Train size: 20000
Val size:   10000
Device:     cuda


In [19]:
best_f1    = 0
best_epoch = 0
for epoch in range(num_epochs):
    print(f"\n{'─'*55}")
    print(f"EPOCH {epoch+1}/{num_epochs}")
    print(f"{'─'*55}")
    train_loss = training(model , train_loader , optimizer , scheduler , device)
    print(f"Train Loss: {train_loss:.4f}")

    # model prformace
    val_loss, val_f1, preds, labels = evaluate(
        model, val_loader, device, label_names
    )
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val F1:     {val_f1*100:.2f}%")
    if val_f1 > best_f1:
        best_f1    = val_f1
        best_epoch = epoch + 1
        model.save_pretrained("./ner-bert-best")
        tokenizer.save_pretrained("./ner-bert-best")




───────────────────────────────────────────────────────
EPOCH 1/3
───────────────────────────────────────────────────────
  Batch 10/313 | Loss: 2.0161
  Batch 20/313 | Loss: 1.9721
  Batch 30/313 | Loss: 1.9069
  Batch 40/313 | Loss: 1.8208
  Batch 50/313 | Loss: 1.7322
  Batch 60/313 | Loss: 1.6491
  Batch 70/313 | Loss: 1.5626
  Batch 80/313 | Loss: 1.4758
  Batch 90/313 | Loss: 1.3880
  Batch 100/313 | Loss: 1.3055
  Batch 110/313 | Loss: 1.2345
  Batch 120/313 | Loss: 1.1693
  Batch 130/313 | Loss: 1.1126
  Batch 140/313 | Loss: 1.0613
  Batch 150/313 | Loss: 1.0189
  Batch 160/313 | Loss: 0.9779
  Batch 170/313 | Loss: 0.9423
  Batch 180/313 | Loss: 0.9070
  Batch 190/313 | Loss: 0.8779
  Batch 200/313 | Loss: 0.8544
  Batch 210/313 | Loss: 0.8293
  Batch 220/313 | Loss: 0.8048
  Batch 230/313 | Loss: 0.7834
  Batch 240/313 | Loss: 0.7649
  Batch 250/313 | Loss: 0.7455
  Batch 260/313 | Loss: 0.7297
  Batch 270/313 | Loss: 0.7146
  Batch 280/313 | Loss: 0.6987
  Batch 290/313 | 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


───────────────────────────────────────────────────────
EPOCH 2/3
───────────────────────────────────────────────────────
  Batch 10/313 | Loss: 0.2725
  Batch 20/313 | Loss: 0.2723
  Batch 30/313 | Loss: 0.2638
  Batch 40/313 | Loss: 0.2561
  Batch 50/313 | Loss: 0.2496
  Batch 60/313 | Loss: 0.2462
  Batch 70/313 | Loss: 0.2467
  Batch 80/313 | Loss: 0.2468
  Batch 90/313 | Loss: 0.2486
  Batch 100/313 | Loss: 0.2506
  Batch 110/313 | Loss: 0.2486
  Batch 120/313 | Loss: 0.2506
  Batch 130/313 | Loss: 0.2509
  Batch 140/313 | Loss: 0.2513
  Batch 150/313 | Loss: 0.2509
  Batch 160/313 | Loss: 0.2530
  Batch 170/313 | Loss: 0.2527
  Batch 180/313 | Loss: 0.2527
  Batch 190/313 | Loss: 0.2513
  Batch 200/313 | Loss: 0.2499
  Batch 210/313 | Loss: 0.2493
  Batch 220/313 | Loss: 0.2490
  Batch 230/313 | Loss: 0.2482
  Batch 240/313 | Loss: 0.2480
  Batch 250/313 | Loss: 0.2473
  Batch 260/313 | Loss: 0.2449
  Batch 270/313 | Loss: 0.2443
  Batch 280/313 | Loss: 0.2434
  Batch 290/313 | 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


───────────────────────────────────────────────────────
EPOCH 3/3
───────────────────────────────────────────────────────
  Batch 10/313 | Loss: 0.1803
  Batch 20/313 | Loss: 0.1836
  Batch 30/313 | Loss: 0.1816
  Batch 40/313 | Loss: 0.1844
  Batch 50/313 | Loss: 0.1881
  Batch 60/313 | Loss: 0.1906
  Batch 70/313 | Loss: 0.1939
  Batch 80/313 | Loss: 0.1980
  Batch 90/313 | Loss: 0.1973
  Batch 100/313 | Loss: 0.1989
  Batch 110/313 | Loss: 0.1982
  Batch 120/313 | Loss: 0.1980
  Batch 130/313 | Loss: 0.1964
  Batch 140/313 | Loss: 0.1970
  Batch 150/313 | Loss: 0.1972
  Batch 160/313 | Loss: 0.1962
  Batch 170/313 | Loss: 0.1960
  Batch 180/313 | Loss: 0.1969
  Batch 190/313 | Loss: 0.1964
  Batch 200/313 | Loss: 0.1972
  Batch 210/313 | Loss: 0.1968
  Batch 220/313 | Loss: 0.1963
  Batch 230/313 | Loss: 0.1966
  Batch 240/313 | Loss: 0.1968
  Batch 250/313 | Loss: 0.1979
  Batch 260/313 | Loss: 0.1966
  Batch 270/313 | Loss: 0.1966
  Batch 280/313 | Loss: 0.1967
  Batch 290/313 | 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
from transformers import AutoModelForTokenClassification
from transformers import AutoTokenizer
import torch

ner_model     = AutoModelForTokenClassification.from_pretrained("./ner-bert-best")
ner_tokenizer = AutoTokenizer.from_pretrained("./ner-bert-best")
ner_model     = ner_model.to(device)
ner_model.eval()

def predict_ner(text):

    inputs = ner_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = ner_model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)
    predictions = predictions[0].cpu().numpy()
    tokens = ner_tokenizer.convert_ids_to_tokens(
        inputs['input_ids'][0].cpu().numpy()
    )

    print(f"\nText: {text}")
    print("─"*50)
    print(f"{'Token':20} {'Label':10}")
    print("─"*50)

    for token, pred in zip(tokens, predictions):
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            continue
        label = ner_model.config.id2label[pred]
        if label != 'O':
            print(f"{token:20} {label:10}")


predict_ner("Elon Musk founded Tesla in California")
predict_ner("Imran Khan was born in Lahore Pakistan")
predict_ner("Microsoft was created by Bill Gates in Seattle")
predict_ner("I am a student at FAST University in Lahore")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Text: Elon Musk founded Tesla in California
──────────────────────────────────────────────────
Token                Label     
──────────────────────────────────────────────────
el                   B-PER     
##on                 B-PER     
mu                   I-PER     
##sk                 I-PER     
tesla                B-ORG     
california           B-LOC     

Text: Imran Khan was born in Lahore Pakistan
──────────────────────────────────────────────────
Token                Label     
──────────────────────────────────────────────────
im                   B-PER     
##ran                I-PER     
khan                 I-PER     
lahore               B-LOC     
pakistan             B-LOC     

Text: Microsoft was created by Bill Gates in Seattle
──────────────────────────────────────────────────
Token                Label     
──────────────────────────────────────────────────
microsoft            B-ORG     
bill                 B-PER     
gates                I-PER     
seatt

In [21]:
import shutil
from google.colab import files

shutil.make_archive(
    "ner-bert-best",
    "zip",
    "./ner-bert-best"
)

# Download
files.download("ner-bert-best.zip")
print("Download started ")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started ✅
